In [1]:
# Import shared loading/cleaning functions from utils.py
from utils import load_clean_data
from utils import get_cv_folds

import pandas as pd
import numpy as np

# Models we will train and compare
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor

# For scaling numeric features (needed for Linear Regression and KNN)
from sklearn.preprocessing import StandardScaler

# For evaluation metrics
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# Note: like Frequency Encoding, Target Encoding isn't built into scikit-learn
# in the exact smoothed form we want, so we implement it manually with pandas.
# This also makes the leakage-prevention and smoothing logic fully visible
# and explainable in your dissertation, rather than hidden inside a library.

In [2]:
# Load the cleaned dataset
df = load_clean_data("data.csv")

# Generate the FIXED set of cross-validation folds
folds = get_cv_folds(df, n_splits=5)

print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Generated {len(folds)} folds")

Loaded 79159 rows, 17 columns
Generated 5 folds


In [3]:
# The target variable we're predicting
y = df["advertised_price"]

# Drop the target from the feature set
X = df.drop(columns=["advertised_price"])

# Categorical columns to encode
categorical_cols = ["make", "model", "generation", "fuel", "body",
                     "transmission", "trim", "plate_age_id"]

# Numeric columns — vehicle_age_days excluded (redundant with vehicle_age_years)
numeric_cols = ["mileage", "list_price", "reg_year", "doors", "engine",
                "engine_missing", "vehicle_age_years"]

print(f"Categorical columns: {len(categorical_cols)}")
print(f"Numeric columns: {len(numeric_cols)}")

Categorical columns: 8
Numeric columns: 7


In [4]:
# Reset results dictionary — storing RMSE, MAE, and R² for each model, per fold
results = {
    "Linear Regression": {"rmse": [], "mae": [], "r2": []},
    "KNN": {"rmse": [], "mae": [], "r2": []},
    "Decision Tree": {"rmse": [], "mae": [], "r2": []},
    "Random Forest": {"rmse": [], "mae": [], "r2": []},
    "Stacking": {"rmse": [], "mae": [], "r2": []}
}

# Smoothing weight for target encoding — controls how many observations a
# category needs before its own mean is trusted over the global mean.
# Chosen as a moderate default given the rare-category findings from
# exploration (e.g. 45% of trim categories had fewer than 5 occurrences)
SMOOTHING_WEIGHT = 10

In [5]:
# Loop through each of the 5 fixed folds
for fold_num, (train_idx, test_idx) in enumerate(folds, start=1):

    # ---- Split data into this fold's train/test portions ----
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # ---- Target Encode the categorical columns (with smoothing) ----
    # Calculated using ONLY training data (X_train and y_train), to avoid
    # leakage — the test set's own prices must never influence the encoding
    # values it receives.
    global_mean = y_train.mean()  # fallback for rare/unseen categories

    X_train_cat = pd.DataFrame(index=X_train.index)
    X_test_cat = pd.DataFrame(index=X_test.index)

    for col in categorical_cols:
        # Build a temporary DataFrame combining this column with the target,
        # using training data only
        temp = pd.DataFrame({col: X_train[col], "target": y_train})

        # For each category: count how many times it appears, and its mean price
        agg = temp.groupby(col)["target"].agg(["mean", "count"])

        # Apply the smoothing formula:
        # smoothed_mean = (count * category_mean + SMOOTHING_WEIGHT * global_mean)
        #                 / (count + SMOOTHING_WEIGHT)
        # Categories with many observations lean toward their own true mean.
        # Categories with few observations lean toward the safer global mean.
        smoothed = (agg["count"] * agg["mean"] + SMOOTHING_WEIGHT * global_mean) / (agg["count"] + SMOOTHING_WEIGHT)

        # Apply this smoothed mapping to both train and test.
        # Any category in test not seen during training becomes NaN via .map(),
        # we fill those with the global mean (the safest possible fallback)
        X_train_cat[col] = X_train[col].map(smoothed).fillna(global_mean)
        X_test_cat[col] = X_test[col].map(smoothed).fillna(global_mean)

    # ---- Combine encoded categorical columns with numeric columns ----
    X_train_final = pd.concat([X_train[numeric_cols], X_train_cat], axis=1)
    X_test_final = pd.concat([X_test[numeric_cols], X_test_cat], axis=1)

    # ---- Scale features for models that need it (Linear Regression, KNN) ----
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_final)
    X_test_scaled = scaler.transform(X_test_final)

    # ---- Define the 5 models ----
    linear_model = LinearRegression()
    knn_model = KNeighborsRegressor()
    tree_model = DecisionTreeRegressor(random_state=42)
    rf_model = RandomForestRegressor(random_state=42, n_estimators=50)

    stacking_model = StackingRegressor(
        estimators=[
            ("linear", LinearRegression()),
            ("knn", KNeighborsRegressor()),
            ("tree", DecisionTreeRegressor(random_state=42)),
            ("rf", RandomForestRegressor(random_state=42, n_estimators=50))
        ],
        final_estimator=LinearRegression(),
        cv=3,
    )

    # ---- Train and evaluate each model ----

    # Linear Regression (scaled features)
    linear_model.fit(X_train_scaled, y_train)
    linear_preds = linear_model.predict(X_test_scaled)
    results["Linear Regression"]["rmse"].append(root_mean_squared_error(y_test, linear_preds))
    results["Linear Regression"]["mae"].append(mean_absolute_error(y_test, linear_preds))
    results["Linear Regression"]["r2"].append(r2_score(y_test, linear_preds))

    # KNN (scaled features)
    knn_model.fit(X_train_scaled, y_train)
    knn_preds = knn_model.predict(X_test_scaled)
    results["KNN"]["rmse"].append(root_mean_squared_error(y_test, knn_preds))
    results["KNN"]["mae"].append(mean_absolute_error(y_test, knn_preds))
    results["KNN"]["r2"].append(r2_score(y_test, knn_preds))

    # Decision Tree (unscaled features)
    tree_model.fit(X_train_final, y_train)
    tree_preds = tree_model.predict(X_test_final)
    results["Decision Tree"]["rmse"].append(root_mean_squared_error(y_test, tree_preds))
    results["Decision Tree"]["mae"].append(mean_absolute_error(y_test, tree_preds))
    results["Decision Tree"]["r2"].append(r2_score(y_test, tree_preds))

    # Random Forest (unscaled features)
    rf_model.fit(X_train_final, y_train)
    rf_preds = rf_model.predict(X_test_final)
    results["Random Forest"]["rmse"].append(root_mean_squared_error(y_test, rf_preds))
    results["Random Forest"]["mae"].append(mean_absolute_error(y_test, rf_preds))
    results["Random Forest"]["r2"].append(r2_score(y_test, rf_preds))

    # Stacking (unscaled features)
    stacking_model.fit(X_train_final, y_train)
    stacking_preds = stacking_model.predict(X_test_final)
    results["Stacking"]["rmse"].append(root_mean_squared_error(y_test, stacking_preds))
    results["Stacking"]["mae"].append(mean_absolute_error(y_test, stacking_preds))
    results["Stacking"]["r2"].append(r2_score(y_test, stacking_preds))

    print(f"Fold {fold_num}/{len(folds)} complete")

print("\nAll folds complete.")

Fold 1/5 complete
Fold 2/5 complete
Fold 3/5 complete
Fold 4/5 complete
Fold 5/5 complete

All folds complete.


In [6]:
# Build a summary table: mean and std of RMSE, MAE, and R² across the folds, per model
summary_rows = []
for model_name, metrics in results.items():
    summary_rows.append({
        "model": model_name,
        "mean_rmse": np.mean(metrics["rmse"]),
        "std_rmse": np.std(metrics["rmse"]),
        "mean_mae": np.mean(metrics["mae"]),
        "std_mae": np.std(metrics["mae"]),
        "mean_r2": np.mean(metrics["r2"]),
        "std_r2": np.std(metrics["r2"])
    })

summary = pd.DataFrame(summary_rows)
summary

,model,mean_rmse,std_rmse,mean_mae,std_mae,mean_r2,std_r2
0,Linear Regression,4329.524686,117.738060,2685.221582,36.838042,0.905184,0.003440
1,KNN,2345.340574,113.618918,1303.239032,24.567449,0.972168,0.002011
2,Decision Tree,2911.844663,156.215965,1537.433996,21.754684,0.956991,0.004565
3,Random Forest,2025.313463,112.560866,1092.738964,16.533597,0.979238,0.001783
4,Stacking,2013.268323,113.529964,1104.129223,19.899495,0.979484,0.001788


In [7]:
# Flatten results into a long-format DataFrame: one row per model per fold per metric
rows = []
for model_name, metrics in results.items():
    for fold_i in range(len(folds)):
        rows.append({
            "encoder": "target",
            "model": model_name,
            "fold": fold_i + 1,
            "rmse": metrics["rmse"][fold_i],
            "mae": metrics["mae"][fold_i],
            "r2": metrics["r2"][fold_i]
        })

results_df = pd.DataFrame(rows)
results_df.to_csv("target_scores.csv", index=False)

print("Saved results to target_scores.csv")

Saved results to target_scores.csv


In [ ]:
# Capture feature importance from Random Forest, same as previous notebooks
feature_importance = pd.DataFrame({
    "feature": X_train_final.columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance["encoder"] = "target"
feature_importance.to_csv("target_feature_importance.csv", index=False)

feature_importance.head(15)

OSError: Cannot save file into a non-existent directory: '..\results'